In [1]:
#---------------------------------------------------------
# 1. LIBRARIES
#---------------------------------------------------------

import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
#---------------------------------------------------------
# 2. DATA PREPROCESSING
#---------------------------------------------------------

import sys
import os

main_dir = os.path.abspath('..')

sys.path.append(main_dir + '/utils')

from preprocessing import preprocess

train_data = pd.read_csv('../data/adult_train_cleaned.csv')

x_train = train_data.drop('income', axis=1)
y_train = train_data['income']

preprocessor = preprocess(x_train)

In [3]:
#---------------------------------------------------------
# 3. HYPERPARAMETER TUNING & CROSS-VALIDATION
#---------------------------------------------------------

x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = LabelEncoder().fit_transform(y_train)

model = LogisticRegression(
    penalty = 'l2',
    class_weight = 'balanced',
    n_jobs = -1,
    random_state = 42
)

param_grid = {
    'max_iter': [1000, 2000, 5000],
    'fit_intercept': [True, False],
    'tol': [0.001, 0.0001, 0.00001, 0.000001],
    'C': [0.1, 1.0, 10],
    'solver': ['lbfgs', 'liblinear'],
}

grid = GridSearchCV(
    estimator = model,
    param_grid = param_grid,
    cv = 5,
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    refit = 'f1',
    n_jobs = -1,
    verbose = 2
)

grid.fit(x_train_preprocessed, y_train_preprocessed)

print(grid.best_params_)
print(grid.best_score_)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
{'C': 10, 'fit_intercept': False, 'max_iter': 1000, 'solver': 'lbfgs', 'tol': 0.0001}
0.6740715876949452


In [4]:
#---------------------------------------------------------
# 4. MODEL EVALUATION ON TEST SET
#---------------------------------------------------------

best_model = grid.best_estimator_

model_pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', best_model)
], memory = None)

model_pipeline.fit(x_train, y_train)

test_data = pd.read_csv('../data/adult_test_cleaned.csv')

x_test = test_data.drop('income', axis=1)
y_test = test_data['income']

y_pred = model_pipeline.predict(x_test)

print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix on Test Set:")
print(confusion_matrix(y_test, y_pred))

Classification Report on Test Set:
              precision    recall  f1-score   support

       <=50K       0.94      0.79      0.86     12430
        >50K       0.55      0.84      0.66      3846

    accuracy                           0.80     16276
   macro avg       0.75      0.81      0.76     16276
weighted avg       0.85      0.80      0.81     16276

Confusion Matrix on Test Set:
[[9803 2627]
 [ 624 3222]]


In [5]:
#---------------------------------------------------------
# 5. MODEL PERSISTENCE
#---------------------------------------------------------

joblib.dump(model_pipeline, '../models/lr_model.joblib')

['../models/lr_model.joblib']